# LinkedIn post analyser

In this notebook, you will build a small Python program that:

1. asks you for a LinkedIn post,
2. asks you for your API key,
3. sends the post to a large language model (LLM), and
4. prints the LLM's analysis of the post: is it engaging, and what value does it give the reader?

Most of the code is already here. Your job is to fill in the gaps, called **mini-tasks**, with help from an AI assistant. You do not need to write the program from scratch, and you do not need to understand every line straight away. Asking good questions about the code is part of the task.

A notebook has two kinds of cells:

- **Text cells**, like this one, explain what to do.
- **Code cells** contain Python code you can run. To run a code cell, click the run button to the left of it, or click inside it and press `Shift + Enter`. The output appears below the cell.

## Step 1: Check that the notebook runs

Run the code cell below. The first time you run a cell, VS Code asks you to choose a kernel: select **Python Environments**, then the recommended Python version, and run the cell again.

If you see `Hello, World!` below the cell, you are ready to start.

In [101]:
print("Hello, World!")

Hello, World!


## Step 2: How to work through the mini-tasks

There are eight mini-tasks. Each one starts with a `# Mini-task` comment in a code cell. For each task:

1. Copy the whole code cell into your AI assistant.
2. Ask it to complete the mini-task in that cell only, and to explain what each line does.
3. Paste its answer back into the same cell, replacing what was there.
4. Run the cell. If you get an error, copy the error message into the assistant and ask what it means.

Assistants sometimes provide code for a different AI service or an older SDK. This notebook uses Google's current `google-genai` Python package and a Gemini API key from Google AI Studio. Keep only the code for the mini-task in its cell so functions are not duplicated.

## Step 3: Install and import the package

Mini-task 1 installs the `google-genai` package into your Codespace. You only need to run it once per Codespace.

In [102]:
# Mini-task 1: install the supported Google Gen AI Python package.
# Tip: in a notebook, install packages with %pip rather than pip.
%pip install -U google-genai


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [103]:
# Mini-task 2: import the Google Gen AI client and hidden-input helper.
from getpass import getpass
import time
from google import genai
from google.genai import types

## Step 4: Write the prompt

The prompt is the instruction you send to the LLM together with the post. Write it so that the LLM **analyses** the post rather than rewriting it. Assistants often suggest a prompt like "Improve this LinkedIn post", which gets you a new post back instead of an analysis, so read the prompt you are given and edit it.

A good prompt says what you want to find out, for example:

- What is the post about, and who is it for?
- Is it engaging? Why or why not?
- What value does it give the reader?
- What one or two changes would make it better?

In [104]:
# Mini-task 3: complete this function so that it returns a prompt asking
# the LLM to analyse the LinkedIn post. The prompt must include the post.
def get_analysis_prompt(linkedin_post):
    prompt = f"""Analyze this LinkedIn post; do not rewrite it.
Describe its topic and likely audience, assess how engaging it is and why,
identify the value it offers readers, and suggest one or two improvements.

Post:
{linkedin_post}"""
    return prompt

## Step 5: Call Gemini

The function below sends your prompt to Gemini and returns its answer. The model name is set in `MODEL_NAME`. Use a model listed in Google's current model documentation and available to your API key:

- [Gemini models](https://ai.google.dev/gemini-api/docs/models)

This notebook uses `gemini-3.8-flash`, the model name suggested by the error you received. If Google reports that it is unavailable to your key, replace it with a Flash model listed in the documentation.

In [109]:
MODEL_NAME = "gemini-3.8-flash"


def get_insights_from_llm(client, linkedin_post):
    prompt = get_analysis_prompt(linkedin_post)
    for attempt in range(2):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
            )
            if not response.text:
                return "Error: Gemini returned an empty response. Check whether the prompt was blocked or filtered."
            return response.text
        except Exception as e:
            error_message = str(e)
            is_overloaded = "503" in error_message and "UNAVAILABLE" in error_message
            is_deadline = "504" in error_message and "DEADLINE_EXCEEDED" in error_message
            if attempt == 0 and (is_overloaded or is_deadline):
                delay = 10 if is_overloaded else 2
                print(f"Gemini is temporarily unavailable; retrying once in {delay} seconds...")
                time.sleep(delay)
                continue
            return f"Error: {type(e).__name__}: {e}"

    return "Error: Gemini request failed after one retry."

## Step 6: Put the program together

The `main()` function asks for a LinkedIn post and a Google AI Studio API key, creates a Google Gen AI client, sends the post to Gemini, and prints the analysis.

**Keep your API key secret.** Never type or paste your API key into a code cell. You will save this notebook to a GitHub repository and share the link for review, and anyone who sees the key can use it and spend your credit. `getpass()` asks for the key in a hidden input box, so it is never saved in the notebook.

In [106]:
def main():
    linkedin_post = input("Paste your LinkedIn post: ")

    # Ask for the API key without displaying or saving it in the notebook.
    api_key = getpass("Google AI Studio API key: ")
    client = genai.Client(
        api_key=api_key,
        http_options=types.HttpOptions(timeout=60000),
    )

    insights = get_insights_from_llm(client, linkedin_post)
    if insights.startswith("Error:"):
        print("\nGemini request failed:")
        print(insights)
        return

    print("\nAnalysis of your LinkedIn post:\n")
    print(insights)


## Step 7: Run the analyser

Run the code cell below. An input box opens at the top of the VS Code window. Paste your LinkedIn post and press `Enter`, then paste your API key and press `Enter`. The analysis appears below the cell.

If you do not have a post to hand, write a fake one, take one from your feed, or use one of your own.

If your post has several paragraphs, the line breaks may be lost when you paste it into the input box. That is fine for this task. If you want to keep the formatting, reading the post from a text file is one of the optional tasks.

If you see `NameError`, Python has forgotten the earlier cells, which happens when you reopen your Codespace or restart the kernel. Click **Run All** in the toolbar above the notebook.

A `504 DEADLINE_EXCEEDED` means Google's server did not finish in time, not necessarily that your key is wrong. The program retries once. If it still fails, wait a little and try again, or choose another Flash model available to your API key. Other errors may indicate a mistyped key, unavailable model, or exhausted quota.

In [107]:
main()


Gemini request failed:
Error: ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


## Step 8: Save your work

Once the analyser works, save the notebook to your GitHub repository as described in the project description. Before you commit, check that your API key does not appear anywhere in the notebook.

Keep a link to your AI assistant conversation as well, because you need to submit it with your repository link. If you used several chats, keep a link to each one.

If you want to go further, pick one of the optional tasks from the project description.